### Load PDF Files

In [28]:
from langchain_community.document_loaders import(
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)


In [29]:
# Method 1 - PyPDFLoader

print("PyPDFloader")

try:
    pypdf_loader = PyPDFLoader("data/pdf/RAG_Notes.pdf")
    pypdf_docs = pypdf_loader.load()
    print(pypdf_docs)
    print(f" Loaded {len(pypdf_docs)} pages")
    print(f" Page 1 content: {pypdf_docs[0].page_content[:100]}...")
    print(f" Metadata: {pypdf_docs[0].metadata}")

except Exception as e: 
    print(f"Error : {e}")

PyPDFloader
[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-03T14:43:13+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-03T14:43:13+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/pdf/RAG_Notes.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="RAG Notes  \x7f  Page 1\n RAG — Retrieval-Augmented Generation\n Concise Study Notes\n1. What is RAG?\n \x7f RAG = Retrieval-Augmented Generation.\n \x7f RAG allows an LLM to use external knowledge while generating an answer.\n \x7f Instead of relying only on the model's trained knowledge, RAG retrieves relevant information from a knowledge\n base.\n\x7f The retrieved information is provided to the LLM as context.\n Basic Flow\n User Query → Retrieve Relevant Information → Add Context → LLM → Final Answer\n2. Why Do We Need RAG?\n \x7f LLM knowledge may become outdated.\n \x7f LLMs d

In [30]:
# Method 2 - PyMuPDFLoader (fast and accurate)
print("\n PyMuLoader")

try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/RAG_Notes.pdf")
    pymupdf_docs = pymupdf_loader.load()

    print(f" Loaded {len(pymupdf_docs)} pages")
    print(f" Includes detailed metadata")
    print(pymupdf_docs)
except Exception as e:
    print(f" Error: {e}")



 PyMuLoader
 Loaded 6 pages
 Includes detailed metadata
[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': 'data/pdf/RAG_Notes.pdf', 'file_path': 'data/pdf/RAG_Notes.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content="RAG Notes  •  Page 1\nRAG — Retrieval-Augmented Generation\nConcise Study Notes\n1. What is RAG?\n• RAG = Retrieval-Augmented Generation.\n• RAG allows an LLM to use external knowledge while generating an answer.\n• Instead of relying only on the model's trained knowledge, RAG retrieves relevant information from a knowledge\nbase.\n• The retrieved information is provided to the LLM as context.\nBasic Flow\nUser Query → Retrieve Relevant Information → Add Context → LLM → Final Answer\n2. Why Do We Need RAG?\n• LLM knowledge may become outdated.\n• LLMs do not automatically know private or company docume

In [31]:
print("PDF Loader Comparison :-")

print("""
1. PyPDFLoader
- Simple and reliable
- Good for most PDFs
- Preserves page numbers
- Basic text extraction
- Use when: Standard text PDFs

2. PyMuPDFLoader
- Fast processing
- Good text extraction
- Image extraction support
- Use when: Speed is important
""")

PDF Loader Comparison :-

1. PyPDFLoader
- Simple and reliable
- Good for most PDFs
- Preserves page numbers
- Basic text extraction
- Use when: Standard text PDFs

2. PyMuPDFLoader
- Fast processing
- Good text extraction
- Image extraction support
- Use when: Speed is important



### Handling PDF Challenges

Purpose of this sections - PDFs are difficult to parse because they:

- store text in complex way (not just simple text)
- can have formatting issues
- may contain scanned images (requiring OCR)
- often have extraction artifacts

In [32]:
# Example of raw PDF extraction

raw_pdf_text = """Python is a popular, high-level, 

    general-purpose programming language known for its simple, 



    readable syntax that resembles natural English.

The python is 100%. (recommended)


Page 1 of 10
"""

# Apply the cleaning function 
def clean_text(text):
    # Remove excessive whitespaces
    text = " ".join(text.split())

    # Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")

    return text

cleaned = clean_text(raw_pdf_text)
print("BEFORE:")
print(repr(raw_pdf_text[:100]))
print("\nAFTER:")
print(repr(cleaned[:100]))


BEFORE:
'Python is a popular, high-level, \n\n    general-purpose programming language known for its simple, \n\n'

AFTER:
'Python is a popular, high-level, general-purpose programming language known for its simple, readable'


In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
from langchain_core.documents import Document
from typing import List

class SmartPDFProcessor:
    """Advance PDF Processing with error handling"""
    def __init__(self, chunk_size = 1000, chunk_overlap = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators = [" "],

        )

    def process_pdf(self, pdf_path:str) -> List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        # Process each page 

        processed_chunks = []

        for page_num, page in enumerate(pages):
            # clean text
            cleaned_text = self._clean_text(page.page_content)

            # skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts = [cleaned_text], 
                metadatas = [{
                    **page.metadata,
                    "page" : page_num + 1,
                    "total_pages" : len(pages),
                    "chunk_method" : "smart_pdf_processor",
                    "char_count" : len(cleaned_text)
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""

        # Remove excessive whitespace
        text = " ".join(text.split())

        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")

        return text


In [38]:
preprocessor = SmartPDFProcessor()


In [ ]:
preprocessor 


In [42]:
# Process a PDF if available

try:
    smart_chunks = preprocessor.process_pdf("data/pdf/RAG_Notes.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"{key} : {value}")

except Exception as e:
    print(f"Processing error: {e}")


Processed into 10 smart chunks

Sample chunk metadata:
producer : ReportLab PDF Library - (opensource)
creator : (unspecified)
creationdate : 2026-09-03T14:43:13+00:00
author : (anonymous)
keywords : 
moddate : 2026-09-03T14:43:13+00:00
subject : (unspecified)
title : (anonymous)
trapped : /False
source : data/pdf/RAG_Notes.pdf
total_pages : 6
page : 1
page_label : 1
chunk_method : smart_pdf_processor
char_count : 1318
